In [1]:
import numpy as np
from typing import Dict, Tuple
from tictactoe import TicTacToeMDP

env = TicTacToeMDP()
all_mdp_states = env.get_all_states()
root_state = (0, 0, 1)  # (x_board, o_board, player_turn)

print(f"Total single-agent decision/terminal states mapped: {len(all_mdp_states)}")

Total single-agent decision/terminal states mapped: 3381


# Dynamic programming (DP)

## Policy iteration
The basic DP approach is policy iteration, which alternates between:
- Policy evaluation: compute value function $V^\pi$ for current policy $\pi$
- Policy improvement: improve current policy $\pi$ with respect to $V^\pi$

### Policy evaluation

Policy evaluation iteratively applies the Bellman expectation equation as an assignment rule across every non-terminal state in our state space:

$$
V_{k+1}(s) \leftarrow \sum_{s^\prime}P(s^\prime \vert s,\pi(s))\left[R(s,\pi(s),s^\prime)+\gamma V_k(s^\prime)\right]
$$

(Where $\gamma=1$ for tictactoe, since it's finite)

Iterates $V_1, V_2, \dots V^\pi$ until the maximum absolute change in values between successive passes falls below a convergence threshold: $\max_{s \in S} \vert V_{k+1}(s)−V_k(s)\vert < \theta$.

In [2]:
def policy_evaluation(
    env: TicTacToeMDP,
    policy: Dict,
    theta: float = 1e-6,
    gamma: float = 1.0,
    V: Dict = None,
) -> Dict:

    # If no initial value distribution is supplied, init from zero
    if V is None:
        V = {s: 0.0 for s in env.get_all_states()}

    while True:
        delta = 0
        for s in env.get_all_states():
            if env.is_terminal(s):
                continue

            v_old = V[s]
            chosen_action = policy[s]

            v_new = 0.0
            for prob, next_state, reward, is_terminal in env.get_transitions(
                s, chosen_action
            ):
                next_val = 0.0 if is_terminal else V[next_state]
                v_new += prob * (reward + gamma * next_val)

            V[s] = v_new
            delta = max(delta, abs(v_old - V[s]))

        if delta < theta:
            break

    return V


# Test policy: pick the first available legal action
baseline_policy = {
    s: env.get_legal_actions(s)[0] for s in all_mdp_states if not env.is_terminal(s)
}
V_baseline = policy_evaluation(env, baseline_policy)
print(
    f"Value of empty board under first-available policy: {V_baseline[root_state]:.4f}"
)

Value of empty board under first-available policy: 0.6042


### Policy Improvement

Using the resulting $V^\pi$, update the current policy to be greedy with respect to the updated value function:

$$
\pi(s) \leftarrow \underset{a \in A(s)}{\operatorname{argmax}} \sum_{s^\prime} P(s^\prime \vert s,a)\left[R(s,a,s^\prime) + \gamma V^\pi(s^\prime)\right]
$$

In [3]:
def policy_iteration(
    env: TicTacToeMDP, theta: float = 1e-6, gamma: float = 0.95
) -> Tuple[Dict, Dict]:
    states = env.get_all_states()

    policy = {s: env.get_legal_actions(s)[0] for s in states if not env.is_terminal(s)}
    V = {s: 0.0 for s in states}

    iteration = 0
    while True:
        iteration += 1

        # 1. Policy eval
        V = policy_evaluation(env, policy, theta, gamma, V=V)

        # 2. Policy Improvement
        policy_stable = True
        for s in states:
            if env.is_terminal(s):
                continue

            old_action = policy[s]
            best_a = None
            best_q = -float("inf")

            for a in env.get_legal_actions(s):
                q_sa = sum(
                    prob * (reward + gamma * (0.0 if is_terminal else V[next_state]))
                    for prob, next_state, reward, is_terminal in env.get_transitions(
                        s, a
                    )
                )

                # Argmax
                if q_sa > best_q:
                    best_q = q_sa
                    best_a = a

            policy[s] = best_a
            if old_action != best_a:
                policy_stable = False

        if policy_stable:
            print(f"Policy Iteration converged after {iteration} loops.")
            break

    return V, policy


V_pi, policy_pi = policy_iteration(env)
print(f"Optimal Value (PI) of empty board: {V_pi[root_state]:.4f}")

Policy Iteration converged after 4 loops.
Optimal Value (PI) of empty board: 0.8905


## Value iteration

Value Iteration combines policy evaluation and improvement by mapping value steps toward optimality by running continuous sweeps that update state-values using the Bellman Optimality Equation:

$$
V_{k+1}(s) \leftarrow max_{a \in A(s)} \sum_{s^\prime} P(s^\prime \vert s,a)\left[R(s,a,s^\prime)+\gamma V_k(s^\prime)\right]
$$

Once the state-value function stabilizes within a margin of θ, a final single-pass greedy extraction sweep is performed to derive the explicit optimal policy π∗.

In [4]:
def value_iteration(
    env: TicTacToeMDP, theta: float = 1e-6, gamma: float = 0.95
) -> Tuple[Dict, Dict]:
    states = env.get_all_states()
    V = {s: 0.0 for s in states}

    iteration = 0
    while True:
        iteration += 1
        delta = 0
        for s in states:
            if env.is_terminal(s):
                continue

            v_old = V[s]
            action_values = []

            for a in env.get_legal_actions(s):
                q_sa = sum(
                    prob * (reward + gamma * (0.0 if is_terminal else V[next_state]))
                    for prob, next_state, reward, is_terminal in env.get_transitions(
                        s, a
                    )
                )
                action_values.append(q_sa)

            V[s] = max(action_values) if action_values else 0.0
            delta = max(delta, abs(v_old - V[s]))

        if delta < theta:
            print(f"Value Iteration converged after {iteration} matrix sweeps.")
            break

    # Extract optimal policy from optimal value function
    optimal_policy = {}
    for s in states:
        if env.is_terminal(s):
            continue
        best_a = None
        best_q = -float("inf")
        for a in env.get_legal_actions(s):
            q_sa = sum(
                prob * (reward + gamma * (0.0 if is_terminal else V[next_state]))
                for prob, next_state, reward, is_terminal in env.get_transitions(s, a)
            )
            if q_sa > best_q:
                best_q = q_sa
                best_a = a

        optimal_policy[s] = best_a

    return V, optimal_policy


V_vi, policy_vi = value_iteration(env)
print(f"Optimal Value (VI) of empty board: {V_vi[root_state]:.4f}")

Value Iteration converged after 4 matrix sweeps.
Optimal Value (VI) of empty board: 0.8905


In [5]:
# Check for optimal policy divergence
mismatches = 0
for s in policy_pi:
    if policy_pi[s] != policy_vi[s]:
        # If policy choices differ, verify that both actions yield identical maximal Q-values
        q_pi = sum(
            p * (r + 0.95 * (0.0 if t else V_vi[ns]))
            for p, ns, r, t in env.get_transitions(s, policy_pi[s])
        )
        q_vi = sum(
            p * (r + 0.95 * (0.0 if t else V_vi[ns]))
            for p, ns, r, t in env.get_transitions(s, policy_vi[s])
        )
        if not np.isclose(q_pi, q_vi):
            mismatches += 1

print(
    f"Total actions showing sub-optimal strategic gaps between PI and VI: {mismatches}"
)

env.play_simulation_game(policy=policy_vi, v_dict=V_vi, human_player=False)

Total actions showing sub-optimal strategic gaps between PI and VI: 0
Initial Board:
   |   |   
---+---+---
   |   |   
---+---+---
   |   |   
Current Value: 0.8905
---------------
Optimal X plays: (0, 0)
 X |   |   
---+---+---
   |   |   
---+---+---
   |   |   
Current Value: 0.0000
---------------
Stochastic Opponent O plays random action: (2, 1)
 X |   |   
---+---+---
   |   |   
---+---+---
   | O |   
Current Value: 0.9421
---------------
Optimal X plays: (0, 2)
 X |   | X 
---+---+---
   |   |   
---+---+---
   | O |   
Current Value: 0.0000
---------------
Stochastic Opponent O plays random action: (1, 2)
 X |   | X 
---+---+---
   |   | O 
---+---+---
   | O |   
Current Value: 1.0000
---------------
Optimal X plays: (0, 1)
 X | X | X 
---+---+---
   |   | O 
---+---+---
   | O |   
Current Value: 0.0000
---------------
Game Over: Optimal Agent X Wins!
